

# 1. Install Java and NLU


In [ ]:
#  https://colab.research.google.com/drive/1DBk55f9iERI9BDA4kmZ8yO6J65jGmcEA?usp=sharing#scrollTo=BAUFklCqLr3V
import os
import pandas as pd
import numpy as np

! apt-get update -qq > /dev/null   
# Install jav
! apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
! pip install nlu pyspark==2.4.7 > /dev/null


In [ ]:
import nlu
pipe = nlu.load('xlnet ')
#embedding = pipe.predict('He was suprised by the diversity of NLU')

xlnet_base_cased download started this may take some time.
Approximate size to download 417.5 MB
[OK!]
sentence_detector_dl download started this may take some time.
Approximate size to download 354.6 KB
[OK!]


# 2. Load data

In [ ]:
from google.colab import files
df=pd.read_csv("./Abstracts-kwlg-Parsed.csv", sep=',')
#df=pd.read_csv("./Methods-Parsed.csv", sep=',')
df=df.rename(columns={'Content_Parsed': 'text'})
df.head()

,Unnamed: 0,Unnamed: 0.1,File_Name,Content,Category,text,Category_Code,keywords
0,0,0,Gapped Blast,Gapped BLAST and PSI-BLAST: a new generation o...,Alignment,gap blast psiblast new generation protein d...,0,blast psiblast protein database blast program...
1,1,1,RapSearch,RAPSearch: a fast protein similarity search to...,Alignment,rapsearch fast protein similarity search tool...,0,rapsearch short reads next generation sequence...
2,2,2,PhenoMeter,PhenoMeter: A Metabolome Database Search Tool ...,Alignment,phenometer metabolome database search tool us...,0,phenometer metabolome database statistical sim...
3,3,3,cuBlASTp,cuBLASTP: Fine-Grained Parallelization of Prot...,Alignment,cublastp finegrained parallelization protein ...,0,cublastp finegrained parallelization protein s...
4,4,4,muBLASTP,muBLASTP: database-indexed protein sequence se...,Alignment,mublastp databaseindexed protein sequence sear...,0,mublastp databaseindexed protein sequence mult...


In [ ]:
import re
import unicodedata
for i in range (len(df['text'])):
  df['text'][i] = unicodedata.normalize('NFKD', df['text'][i]).encode('ascii', 'ignore').decode("utf-8")
  df['text'][i] = re.sub(r'[^\w]', ' ', df['text'][i])


#  3. Load Model and Embed sample string

In [ ]:
# # dfx=pd.DataFrame()
# # dfx['text']= ["I do not like cherries", "he is my little cherry"]
# # dfx.head()
# embedding = pipe.predict(["I do not like cherries"])
# embedding

In [ ]:
import numpy as np
embeddings = []
all_embeddings = []
sentence_embedding = np.empty(768, dtype=object)
c=0

for txt in df['text']:
  print("*****************item ",c)
  embedding = pipe.predict([txt])
  #embedding = embedding.reset_index(drop=True)
  flatkeys=[element for sublist in embedding['token'].tolist() for element in sublist]
  flatvalues=[element for sublist in embedding['word_embedding_xlnet'].tolist() for element in sublist]
  embedding = dict(zip(flatkeys, flatvalues))
  embeddings.append(embedding)
  c=c+1


for l in range(len(embeddings)):
  for v in embeddings[l].values():
    sentence_embedding=np.vstack((sentence_embedding, v))
  sentence_embedding = np.delete(sentence_embedding, obj=0, axis=0)
  sentence_embedding = (np.mean(sentence_embedding, axis=0)).tolist()
  all_embeddings.append(sentence_embedding)

all_embeddings = np.array(all_embeddings)
all_embeddings = pd.DataFrame(all_embeddings)

all_embeddings.insert(loc=0, column='text', value=df['text'])

*****************item  0
*****************item  1
*****************item  2
*****************item  3
*****************item  4
*****************item  5
*****************item  6
*****************item  7
*****************item  8
*****************item  9
*****************item  10
*****************item  11
*****************item  12
*****************item  13
*****************item  14
*****************item  15
*****************item  16
*****************item  17
*****************item  18
*****************item  19
*****************item  20
*****************item  21
*****************item  22
*****************item  23
*****************item  24
*****************item  25
*****************item  26
*****************item  27
*****************item  28
*****************item  29
*****************item  30
*****************item  31
*****************item  32
*****************item  33
*****************item  34
*****************item  35
*****************item  36
*****************item  37
*****************item 

In [ ]:
print(all_embeddings.shape)
print(all_embeddings)
all_embeddings.to_csv('Abstracts-kwlg-XLNET.csv') 
files.download("Abstracts-kwlg-XLNET.csv")

(224, 769)
                                                  text  ...       767
0    gap blast  psiblast  new generation  protein d...  ... -1.152114
1    rapsearch  fast protein similarity search tool...  ... -1.211162
2    phenometer  metabolome database search tool us...  ... -1.148590
3    cublastp finegrained parallelization  protein ...  ... -1.302649
4    mublastp databaseindexed protein sequence sear...  ... -1.353424
..                                                 ...  ...       ...
219  quast quality assessment tool  genome assembli...  ... -2.967609
220  versatile genome assembly evaluation  quastlg ...  ... -2.163498
221  busco assess genome assembly  annotation compl...  ... -1.766361
222  dnaqet  framework  compute  consolidate metric...  ... -1.735496
223  laser large genome assembly evaluator genome a...  ... -1.178139

[224 rows x 769 columns]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>